In [1]:
import pandas as pd
from time import sleep
import datetime
import os
from bs4 import BeautifulSoup
from pandas import ExcelWriter
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
from time import sleep

print("Running HU CBH Web Scraping Tool v.1.0")
regulatorName = ''
#scriptfolder=os.path.dirname(os.path.abspath(__file__))
scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

os.chdir(scriptfolder)

now=datetime.datetime.now()
filename= 'HU CBH data {}.xlsx'.format(str(now).replace(":",".")[:-7])


#The following Regcodes are excluded from the reglist: 'HU CBH 4', 'HU CBH 18'


regdict= {'HU CBH 1': ['//*[@id="-6"]/div[2]', '//*[@id="-20"]/div[2]']
        , 'HU CBH 2': ['//*[@id="-6"]/div[2]', '//*[@id="-19"]/div[2]']
        , 'HU CBH 3': ['//*[@id="-8"]/div[2]', '//*[@id="-25"]/div[2]']
        , 'HU CBH 4': ['', '']
        , 'HU CBH 5': ['//*[@id="-8"]/div[2]', '//*[@id="132"]/div[2]']
        , 'HU CBH 6': ['//*[@id="-8"]/div[2]', '//*[@id="-24"]/div[2]']
        , 'HU CBH 7': ['//*[@id="-1"]/div[2]', '//*[@id="-10"]/div[2]']
        , 'HU CBH 8': ['//*[@id="-1"]/div[2]', '//*[@id="-9"]/div[2]']
        , 'HU CBH 9': ['//*[@id="-7"]/div[2]', '//*[@id="39"]/div[2]']
        , 'HU CBH 10': ['//*[@id="-7"]/div[2]', '//*[@id="-23"]/div[2]']
        , 'HU CBH 11': ['//*[@id="-5"]/div[2]', '//*[@id="-17"]/div[2]']
        , 'HU CBH 12': ['//*[@id="-5"]/div[2]', '//*[@id="-18"]/div[2]']
        , 'HU CBH 13': ['//*[@id="-3"]/div[2]', '//*[@id="-13"]/div[2]']
        , 'HU CBH 14': ['//*[@id="-3"]/div[2]', '//*[@id="-15"]/div[2]']
        , 'HU CBH 15': ['//*[@id="-3"]/div[2]', '//*[@id="-14"]/div[2]']
        , 'HU CBH 16': ['//*[@id="-3"]/div[2]', '//*[@id="-12"]/div[2]']
        , 'HU CBH 17': ['//*[@id="-4"]/div[2]', '//*[@id="-16"]/div[2]']
        , 'HU CBH 18': ['//*[@id="-4"]/div[2]', '']}

driver = webdriver.Chrome()
driver.maximize_window()

catlist=['Name', 'Previous name', 'Administrative address', 'Type of institution', 'Registration number', 'Registry court/court number', 
        'Website address', 'Location of publication', 'Legal status', '[EN]Közérdeklődésre számon tartott hitelintézet']


xname=[]
xpname=[]
xadmadd=[]
xtypeinst=[]
xregnum=[]
xregcourt=[]
xweb=[]
xlocpub=[]
xlegsta=[]
xen=[]
xauth=[]
clinks=[]
reglist = []

for reg in regdict:
    print('Working with {}.'.format(reg))

    driver.get('https://intezmenykereso.mnb.hu/en/Home/Index')
    sleep(3)
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    # Define a standard timeout (e.g., 10 seconds)
    wait = WebDriverWait(driver, 10)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight / 2);")

    # 1. Wait until element is visible and clickable, then click
    element_1 = wait.until(EC.element_to_be_clickable((By.XPATH, '/html/body/div[3]/div[6]')))
    element_1.click()

    # 2. Click the complex search panel
    wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="complex-search-panel"]/div[1]'))).click()

    # 3. Scroll to a specific element and click
    # We use execute_script to bring the element into the center of the view
    sleep(3)
    target_element = wait.until(EC.presence_of_element_located((By.XPATH, regdict[reg][0])))
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", target_element)
    target_element.click()

    # 4. Handle the remaining clicks with waits instead of sleep
    wait.until(EC.element_to_be_clickable((By.XPATH, regdict[reg][1]))).click()
    wait.until(EC.element_to_be_clickable((By.XPATH, '//*[@id="complex-inst-search-button"]'))).click()
    # while True:

    #     captcha=input('Solved Captcha? (Y/N): ')

    #     if captcha=='Y':

    #         break

    #     else:

    #         print('Type Y when the captcha has been manually solved.')



    #driver.switch_to.default_content()

    source=driver.page_source

    tempstr=BeautifulSoup(source, 'html.parser')

    count=int(tempstr.find('text', {'id':'count'}).text)

    tcount=int(tempstr.find('text', {'id':'total-count'}).text)

    while count<tcount:

        driver.execute_script("var scrollingElement = (document.scrollingElement || document.body);scrollingElement.scrollTop = scrollingElement.scrollHeight;")

        sleep(0.1)

        driver.find_element(By.XPATH, '//body').send_keys(Keys.CONTROL+Keys.END)

        try:

            driver.find_element(By.XPATH, '//*[@id="result-table"]/div[5]/div[1]/input').click()

        except:

            sleep(1)

            source=driver.page_source

            tempstr=BeautifulSoup(source, 'html.parser')

            count=int(tempstr.find('text', {'id':'count'}).text)

    source=driver.page_source

    tempstr=BeautifulSoup(source, 'html.parser')

    rows=tempstr.find_all('div',  {'class':'result-table-row'})

    for row in rows:

        auth=row.find_all("div",  {"class":"result-table-cell"})[3]

        auth=auth.find('input', value= True)['value'].strip()

        if 'Not' not in auth:

            auth=auth.split(' ')[0].strip()

        xauth.append(auth)

        row=row.find_all("div",  {"class":"result-table-cell"})[5]

        lid=''

        lid=row.find('input', lid=True)['lid']

        clinks.append(lid)

    for link in range(len(clinks)):

        print('Working with firm {} of {} firms. (Reg code: {})'.format(link+1, len(clinks), reg))

        tempdict={}

        page='https://intezmenykereso.mnb.hu/en/Details/Index?LId='+clinks[link]+'&EntityType=Institute&expandAccordions=IntezmenyAlapadatok'

        driver.get(page)

        sleep(0.25)

        soup=BeautifulSoup(driver.page_source, 'html.parser')

        datadiv=soup.find('div', {"id":"details-list"})

        if datadiv!=None:

            data=datadiv.find_all('div',  {'class':"result-table-row"})

        else:

            data=[]

            print('datadiv is None.')

        for ro in data:

            ro=ro.find_all("div",  {"class":"result-table-cell"})

            tempdict[ro[0].text.strip()]=ro[1].text.strip()

        for ca in catlist:

            if ca not in tempdict:

                tempdict[ca]=''

            else:

                pass

        xname.append(tempdict[catlist[0]])

        xpname.append(tempdict[catlist[1]])

        xadmadd.append(tempdict[catlist[2]])

        xtypeinst.append(tempdict[catlist[3]])

        xregnum.append(tempdict[catlist[4]])

        xregcourt.append(tempdict[catlist[5]])

        xweb.append(tempdict[catlist[6]])

        xlocpub.append(tempdict[catlist[7]])

        xlegsta.append(tempdict[catlist[8]])

        xen.append(tempdict[catlist[9]])

        reglist.append(reg)
                        

df= pd.DataFrame({'Reg List': reglist, 'Name': xname, 'Previous name': xpname, 'Administrative address': xadmadd,'Authorization': xauth, 'Type of institution': xtypeinst, 'Registration number': xregnum, 'Registry court/court number': xregcourt, 'Website address': xweb, 'Location of publication': xlocpub, 'Legal status': xlegsta, '[EN]Közérdeklődésre számon tartott hitelintézet': xen})
writer = ExcelWriter(filename)
df.to_excel(writer, reg)


writer.save()
writer.close()

sleep(3)


driver.quit()





    
    

Running HU CBH Web Scraping Tool v.1.0
Working with HU CBH 1.


ElementClickInterceptedException: Message: element click intercepted: Element is not clickable at point (632, 632)
  (Session info: chrome=143.0.7499.147); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementclickinterceptedexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x7ff7308288e5
	0x7ff730828940
	0x7ff73060165d
	0x7ff7306619e8
	0x7ff73065f40f
	0x7ff73065c417
	0x7ff73065b338
	0x7ff73064ce7f
	0x7ff730681fda
	0x7ff73064c746
	0x7ff7306aac97
	0x7ff73064ac29
	0x7ff73064ba93
	0x7ff730b40640
	0x7ff730b3af80
	0x7ff730b596e6
	0x7ff730845de4
	0x7ff73084ed8c
	0x7ff730832004
	0x7ff7308321b5
	0x7ff730817ee2
	0x7ffb2b0e259d
	0x7ffb2c0eaf78
